# 02. Technical Indicators & Chart Analysis
Deep dive into computing and plotting RSI, MACD, Exponential Moving Averages (EMA 9, 21, 50, 200), Bollinger Bands, and Volume Spikes for any Binance symbol.

In [1]:
import sys
from pathlib import Path
project_root = Path.cwd() if (Path.cwd() / "backend").exists() else Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from backend.binance_client import binance_client
from backend.indicators import enrich_klines_dataframe

## 1. Fetch & Enrich Kline Data (ETH/USDT)

In [2]:
symbol = 'ETHUSDT'
interval = '1h'
klines = binance_client.get_klines(symbol, interval=interval, limit=120)
df = enrich_klines_dataframe(klines)
df.tail(5)

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_asset_volume,...,ema_9,ema_21,ema_50,ema_200,bb_middle,bb_upper,bb_lower,bb_bandwidth,atr,rvol
115,2026-07-21 12:00:00,1935.69,1945.23,1932.38,1942.79,15279.9073,2026-07-21 12:59:59.999,2.964903e+07,146292,7634.19410000,...,1933.674307,1919.625032,1898.225419,1873.744308,1919.9410,1952.506667,1887.375333,3.392361,13.396866,1.119530
116,2026-07-21 13:00:00,1942.80,1943.77,1927.53,1940.30,19914.6022,2026-07-21 13:59:59.999,3.855937e+07,170585,8379.26910000,...,1934.999446,1921.504575,1899.875403,1874.406554,1921.5670,1954.816311,1888.317689,3.460645,13.599947,1.428341
117,2026-07-21 14:00:00,1940.31,1944.00,1928.00,1931.87,14705.6180,2026-07-21 14:59:59.999,2.847468e+07,162854,7028.06910000,...,1934.373557,1922.446886,1901.130093,1874.978330,1923.1845,1955.035997,1891.333003,3.312370,13.771379,1.066504
118,2026-07-21 15:00:00,1931.88,1932.51,1923.36,1931.81,12511.0061,2026-07-21 15:59:59.999,2.412001e+07,155853,5811.50330000,...,1933.860845,1923.298078,1902.333226,1875.543819,1924.8520,1954.681152,1895.022848,3.099371,13.441281,0.899154
119,2026-07-21 16:00:00,1931.81,1935.36,1924.14,1924.34,3603.2287,2026-07-21 16:59:59.999,6.954125e+06,48269,1707.59680000,...,1931.956676,1923.392798,1903.196237,1876.029353,1925.8270,1954.138115,1897.515885,2.940151,13.282618,0.260603


## 2. Interactive Candlestick + RSI + MACD Subplots

In [3]:
fig = make_subplots(
    rows=3, cols=1, 
    shared_xaxes=True,
    vertical_spacing=0.03,
    row_heights=[0.6, 0.2, 0.2],
    subplot_titles=(f'{symbol} Price & EMA Overlay', 'RSI (14)', 'MACD (12, 26, 9)')
)

# Main Candlestick
fig.add_trace(go.Candlestick(
    x=df['open_time'],
    open=df['open'], high=df['high'], low=df['low'], close=df['close'],
    name='Candles'
), row=1, col=1)

# EMA 9 & EMA 21
fig.add_trace(go.Scatter(x=df['open_time'], y=df['ema_9'], name='EMA 9', line=dict(color='cyan', width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=df['open_time'], y=df['ema_21'], name='EMA 21', line=dict(color='orange', width=1)), row=1, col=1)

# RSI
fig.add_trace(go.Scatter(x=df['open_time'], y=df['rsi'], name='RSI', line=dict(color='purple', width=1.5)), row=2, col=1)
fig.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=1)

# MACD
fig.add_trace(go.Scatter(x=df['open_time'], y=df['macd'], name='MACD', line=dict(color='blue', width=1)), row=3, col=1)
fig.add_trace(go.Scatter(x=df['open_time'], y=df['macd_signal'], name='Signal', line=dict(color='orange', width=1)), row=3, col=1)
colors = ['green' if val >= 0 else 'red' for val in df['macd_hist']]
fig.add_trace(go.Bar(x=df['open_time'], y=df['macd_hist'], name='Histogram', marker_color=colors), row=3, col=1)

fig.update_layout(height=800, title_text=f"{symbol} Technical Analysis View", template='plotly_dark')
fig.show()